# STA272 Lab #6

## Purpose of Lab

Welcome to your sixth weekly tutorial lab!

You are encouraged to refer to lecture content and liberally use course resources.

## Logistics

Due date: The homework is due **11:59pm on Thursday, February 12, 2026.**

You will submit your homework on [MarkUs](https://markus.teach.cs.toronto.edu/markus/).

1. Download this file (`STA272_lab6_student.ipynb`) from JupyterHub. (See [our JupyterHub Guide](../guides/jupyterhub_guide.ipynb) for detailed instructions.)
2. Submit this file to MarkUs under the hw6 assignment. (See [our MarkUs Guide](../guides/markus_guide.ipynb) for detailed instructions.)

## Random Forest for Multi-Class Classification

In this lab we will fit a random forest model to predict a categorical outcome using the CPS employment data for recent graduates.

**Research Question:** Can we predict the worker category of recent graduates (ages 22–30 with a bachelor's degree or higher)?

The target variable `worker_cat` has five categories:

1. Unemployed
2. Part-time
3. Private for-profit
4. Government
5. Self-employed

## Task #1

The file `cps_empl.csv` contains individual-level microdata from the IPUMS CPS for recent graduates (ages 22–30 with a bachelor's degree or higher). Load this file into a pandas dataframe called `cps_df`.

Print the shape of the dataframe and display the first few rows.

In [ ]:
# Place your answer for Task #1 in this cell


In [ ]:
# Answer cell

import pandas as pd
import numpy as np

cps_df = pd.read_csv('cps_empl.csv')
print(f"Shape: {cps_df.shape}")
cps_df.head()

## Task #2

Create a horizontal bar plot of the target variable `worker_cat` showing the count of each category. Use `value_counts()` and the pandas `.plot(kind='barh')` method.

Fill in `...` in the code below.

In [ ]:
# Place your answer for Task #2 in this cell

import matplotlib.pyplot as plt

...

plt.xlabel('Count')
plt.title('Distribution of Worker Categories')
plt.tight_layout()
plt.show()

In [ ]:
# Answer cell

import matplotlib.pyplot as plt

cps_df['worker_cat'].value_counts().plot(kind='barh')

plt.xlabel('Count')
plt.title('Distribution of Worker Categories')
plt.tight_layout()
plt.show()

## Task #3

The variable `worker_cat` is the target variable for multi-class classification with five categories: Unemployed, Part-time, Private for-profit, Government, and Self-employed.

Since `race_cat` is a character variable, we need to convert it to numeric dummy variables for scikit-learn. Use `pd.get_dummies()` with `drop_first=True` to encode `race_cat` and store the result in `cps_encoded`.

Then create:
- `X`: a dataframe with the following feature columns: `AGE`, `female`, `married`, `educ_cat`, `faminc_cat`, `metro_binary`, `us_citizen`, `has_children`, `insured`, `INCWAGE`, `FIRMSIZE`, `NCHILD`, `YEAR`, `race_cat_Black`, `race_cat_Other`, `race_cat_White`
- `y`: the `worker_cat` column from `cps_encoded`

In [ ]:
# Place your answer for Task #3 in this cell

cps_encoded = ...

feature_cols = ['AGE', 'female', 'married', 'educ_cat', 'faminc_cat',
                'metro_binary', 'us_citizen', 'has_children', 'insured',
                'INCWAGE', 'FIRMSIZE', 'NCHILD', 'YEAR',
                'race_cat_Black', 'race_cat_Other', 'race_cat_White']

X = ...
y = ...

In [ ]:
# Answer cell

cps_encoded = pd.get_dummies(cps_df, columns=['race_cat'], drop_first=True)

feature_cols = ['AGE', 'female', 'married', 'educ_cat', 'faminc_cat',
                'metro_binary', 'us_citizen', 'has_children', 'insured',
                'INCWAGE', 'FIRMSIZE', 'NCHILD', 'YEAR',
                'race_cat_Black', 'race_cat_Other', 'race_cat_White']

X = cps_encoded[feature_cols]
y = cps_encoded['worker_cat']

## Task #4

Create training and test sets using `train_test_split` from `sklearn.model_selection` with:
- test size of 20%
- `random_state=272`
- `stratify=y` to maintain class proportions

Store the results as `X_train`, `X_test`, `y_train`, `y_test`.

Print the training set class distribution using `y_train.value_counts()`.

In [ ]:
# Place your answer for Task #4 in this cell


In [ ]:
# Answer cell

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=272, stratify=y)

print(f"Training set: {len(X_train):,}")
print(f"Test set:     {len(X_test):,}")
print()
print("Training set class distribution:")
print(y_train.value_counts())

**Question**: Do you notice any class imbalance in the training set?

## Task #5

First, try different values of `n_estimators` from the list `[1, 5, 10, 25, 50, 100, 200, 500]` 
and plot test accuracy vs. number of trees. Use the following parameters:
- `max_features='sqrt'`
- `class_weight='balanced'`
- `random_state=272`

Based on the plot, fit a final `RandomForestClassifier` with `n_estimators=100` and store it in `worker_rf`. 
Print the training and test accuracy.

In [ ]:
# Place your answer for Task #5 in this cell

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

n_trees_list = [1, 5, 10, 25, 50, 100, 200, 500]
test_acc_list = []

for n_trees in n_trees_list:
    rf = ...
    ...
    test_acc_list.append(...)

plt.figure(figsize=(8, 4), dpi=150)
plt.plot(...)
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('Test Accuracy')
plt.title('Random Forest: Test Accuracy vs. Number of Trees')
plt.tight_layout()
plt.show()

# Fit the final model with n_estimators=100
worker_rf = ...
worker_rf.fit(...)

print(f'Training accuracy: {accuracy_score(y_train, worker_rf.predict(X_train)):.3f}')
print(f'Test accuracy:     {accuracy_score(y_test, worker_rf.predict(X_test)):.3f}')

In [ ]:
# Answer cell

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

n_trees_list = [1, 5, 10, 25, 50, 100, 200, 500]
test_acc_list = []

for n_trees in n_trees_list:
    rf = RandomForestClassifier(
        n_estimators=n_trees,
        max_features='sqrt',
        class_weight='balanced',
        random_state=272
    )
    rf.fit(X_train, y_train)
    test_acc_list.append(accuracy_score(y_test, rf.predict(X_test)))

plt.figure(figsize=(8, 4), dpi=150)
plt.plot(n_trees_list, test_acc_list, 'o-', markersize=5)
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('Test Accuracy')
plt.title('Random Forest: Test Accuracy vs. Number of Trees')
plt.tight_layout()
plt.show()

# Fit the final model with n_estimators=100
worker_rf = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    class_weight='balanced',
    random_state=272
)
worker_rf.fit(X_train, y_train)

print(f'Training accuracy: {accuracy_score(y_train, worker_rf.predict(X_train)):.3f}')
print(f'Test accuracy:     {accuracy_score(y_test, worker_rf.predict(X_test)):.3f}')

**Question**: Is using 100 trees a good choice?

## Task #6

Compute the predicted classes on the test set and store them in `y_pred`. Then compute:
- the confusion matrix using `confusion_matrix` and store it in `cmatrix`
- the classification report using `classification_report` and store it in `creport`

Print both results. Which worker categories does the random forest predict well? Which does it struggle with?

In [ ]:
# Place your answer for Task #6 in this cell

from sklearn.metrics import confusion_matrix, classification_report

y_pred = ...

cmatrix = ...

creport = ...

print(cmatrix)
print(creport)

In [ ]:
# Answer cell

from sklearn.metrics import confusion_matrix, classification_report

y_pred = worker_rf.predict(X_test)

cmatrix = confusion_matrix(y_test, y_pred, labels=worker_rf.classes_)

creport = classification_report(y_test, y_pred, digits=3)

print("Confusion Matrix:")
cm_df = pd.DataFrame(cmatrix, index=worker_rf.classes_, columns=worker_rf.classes_)
print(cm_df)
print()
print(creport)

**Question:** Which worker categories does the random forest predict well? Which does it struggle with?

## Task #7

Compute and plot the variable importance from the random forest model `worker_rf`.

Create a dataframe called `importance_df` with columns `Feature` and `Importance`, sorted by importance in descending order. Then create a horizontal bar plot.

Fill in `...` in the code below.

In [ ]:
# Place your answer for Task #7 in this cell

importances = ...
feature_names = ...

importance_df = (pd.DataFrame({'Feature': feature_names, 'Importance': importances})
                 .sort_values('Importance', ascending=False))

print(importance_df.to_string(index=False))

plt.figure(figsize=(8, 5), dpi=150)
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.xlabel('Importance (Mean Decrease in Gini)')
plt.title('Random Forest: Variable Importance (Worker Categories)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Answer cell

importances = worker_rf.feature_importances_
feature_names = worker_rf.feature_names_in_

importance_df = (pd.DataFrame({'Feature': feature_names, 'Importance': importances})
                 .sort_values('Importance', ascending=False))

print(importance_df.to_string(index=False))

plt.figure(figsize=(8, 5), dpi=150)
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.xlabel('Importance (Mean Decrease in Gini)')
plt.title('Random Forest: Variable Importance (Worker Categories)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

**Question**: Give a brief interpretation of the most important variable in predicting worker categories